In [3]:
import pandas as pd

In [4]:
df = pd.read_csv("../data/raw/events.csv")

In [5]:
df.shape

(2756101, 5)

In [6]:
df.head()

,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2756101 entries, 0 to 2756100
Data columns (total 5 columns):
 #   Column         Dtype  
---  ------         -----  
 0   timestamp      int64  
 1   visitorid      int64  
 2   event          str    
 3   itemid         int64  
 4   transactionid  float64
dtypes: float64(1), int64(3), str(1)
memory usage: 105.1 MB


In [8]:
df.isna().sum()

timestamp              0
visitorid              0
event                  0
itemid                 0
transactionid    2733644
dtype: int64

In [9]:
df.nunique()

timestamp        2750455
visitorid        1407580
event                  3
itemid            235061
transactionid      17672
dtype: int64

In [10]:
df['event'].value_counts()

event
view           2664312
addtocart        69332
transaction      22457
Name: count, dtype: int64

In [11]:
df['datetime'] = pd.to_datetime(df['timestamp'], unit = 'ms')

In [12]:
df.head()

,timestamp,visitorid,event,itemid,transactionid,datetime
0,1433221332117,257597,view,355908,NaN,2015-06-02 05:02:12.117
1,1433224214164,992329,view,248676,NaN,2015-06-02 05:50:14.164
2,1433221999827,111016,view,318965,NaN,2015-06-02 05:13:19.827
3,1433221955914,483717,view,253185,NaN,2015-06-02 05:12:35.914
4,1433221337106,951259,view,367447,NaN,2015-06-02 05:02:17.106


In [13]:
df['datetime'].max()

Timestamp('2015-09-18 02:59:47.788000')

In [14]:
df['datetime'].min()

Timestamp('2015-05-03 03:00:04.384000')

### Сколько событий в среднем приходится на одного пользователя?

In [15]:
len(df)/df['visitorid'].nunique()

1.9580421716705267

### Количество событий для каждого visitorid

In [16]:
df['visitorid'].value_counts()

visitorid
1150086    7757
530559     4328
152963     3024
895999     2474
163561     2410
           ... 
1392454       1
226214        1
591435        1
762376        1
1184451       1
Name: count, Length: 1407580, dtype: int64

In [17]:
events_per_user = df['visitorid'].value_counts()

In [18]:
events_per_user.describe()

count    1.407580e+06
mean     1.958042e+00
std      1.258049e+01
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      2.000000e+00
max      7.757000e+03
Name: count, dtype: float64

### Какой процент пользователей совершил только одно событие за весь период?

In [19]:
events_per_user = events_per_user.reset_index()

In [20]:
event_user1 = events_per_user[events_per_user['count'] == 1]

In [21]:
len(event_user1)/len(events_per_user)*100

71.15474786513022

In [22]:
result = df[df['visitorid'].isin(event_user1['visitorid'])]
print(result)

             timestamp  visitorid event  itemid  transactionid  \
4        1433221337106     951259  view  367447            NaN   
10       1433222162373     176446  view   10572            NaN   
16       1433223239808    1377281  view  251467            NaN   
18       1433224244282    1370216  view  176721            NaN   
23       1433221275206     864246  view   36642            NaN   
...                ...        ...   ...     ...            ...   
2756083  1438401803214    1392454  view    4148            NaN   
2756093  1438400498250     226214  view  190530            NaN   
2756096  1438398785939     591435  view  261427            NaN   
2756097  1438399813142     762376  view  115946            NaN   
2756099  1438398530703    1184451  view  283392            NaN   

                       datetime  
4       2015-06-02 05:02:17.106  
10      2015-06-02 05:16:02.373  
16      2015-06-02 05:33:59.808  
18      2015-06-02 05:50:44.282  
23      2015-06-02 05:01:15.206  
...

In [23]:
result['event'].value_counts(normalize=True) * 100

event
view           99.739706
addtocart       0.253305
transaction     0.006989
Name: proportion, dtype: float64

In [24]:
result[result['event'] == 'transaction'].sort_values(by = 'datetime')

,timestamp,visitorid,event,itemid,transactionid,datetime
1511731,1430881103410,409939,transaction,465622,17623.0,2015-05-06 02:58:23.410
1560451,1431030300577,1320501,transaction,369447,8430.0,2015-05-07 20:25:00.577
1578851,1431120145258,1286040,transaction,386160,17033.0,2015-05-08 21:22:25.258
1641421,1431454309756,874668,transaction,320181,662.0,2015-05-12 18:11:49.756
1640918,1431496383299,682501,transaction,17478,11766.0,2015-05-13 05:53:03.299
...,...,...,...,...,...,...
1328374,1441931262832,926439,transaction,309336,4500.0,2015-09-11 00:27:42.832
1347320,1441942008811,41416,transaction,107017,15120.0,2015-09-11 03:26:48.811
1383303,1442108902572,837376,transaction,247427,1373.0,2015-09-13 01:48:22.572
1447818,1442430455008,1124699,transaction,99601,16381.0,2015-09-16 19:07:35.008


### Сколько уникальных пользователей совершало каждый тип события?

In [25]:
df.groupby('event')['visitorid'].nunique()

event
addtocart        37722
transaction      11719
view           1404179
Name: visitorid, dtype: int64

### Сколько пользователей одновременно имеют хотя бы один view и хотя бы один addtocart?

In [26]:
view_users = df[df['event'] == 'view']['visitorid'].unique()

In [27]:
cart_users = df[df['event'] == 'addtocart']['visitorid'].unique()

In [28]:
len(set(view_users) & set(cart_users))

34401

### Проверка дубликатов

In [31]:
df.duplicated().sum()

np.int64(460)

In [32]:
df[df.duplicated(keep=False)]

,timestamp,visitorid,event,itemid,transactionid,datetime
13734,1433180781440,1045411,addtocart,379647,NaN,2015-06-01 17:46:21.440
22404,1433180781440,1045411,addtocart,379647,NaN,2015-06-01 17:46:21.440
24476,1433274223925,366538,addtocart,252068,NaN,2015-06-02 19:43:43.925
25508,1433265100661,198153,addtocart,48715,NaN,2015-06-02 17:11:40.661
27095,1433270868154,1268755,view,60980,NaN,2015-06-02 18:47:48.154
...,...,...,...,...,...,...
2748964,1438346018813,722488,view,97303,NaN,2015-07-31 12:33:38.813
2750464,1438370523152,896869,addtocart,85004,NaN,2015-07-31 19:22:03.152
2752725,1438405570135,708442,addtocart,392883,NaN,2015-08-01 05:06:10.135
2753763,1438376127329,372829,addtocart,388276,NaN,2015-07-31 20:55:27.329


In [37]:
df = df.drop_duplicates()

In [38]:
df.duplicated().sum()

np.int64(0)

In [39]:
df.shape

(2755641, 6)